In [35]:
#imports
from openai import OpenAI
from scraper import fetch_website_contents,fetch_website_links
import json
from IPython.display import Markdown,display,update_display
from urllib.parse import urljoin, urlparse

In [36]:
# setting up the environment
ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

In [37]:
# system prompt for link
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevent to include in a brochure about a company,
such as links to an about page, or a company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links":[
        {"type":"about page", "url": "https://whatever/about"}
        {"type": "career page", "url": "https://whatever/career"}
    ]
}
"""

In [38]:
# user prompt for links
def get_links_user_prompt(url):
    user_prompt = f"""
here is the list of links on the website {url}
please decide which of these are the relevent web links for a brochure about the company,
respond with the full https URL in JSON format
Do not include Terms of Service, Privacy, email links.

Links(some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [39]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model="gemma3",
        messages=[
            {"role":"system","content":link_system_prompt},
            {"role":"user","content":get_links_user_prompt(url)}
        ],
        response_format={"type":"json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
     # Fix any relative URLs the model returned
    base = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
    for link in links.get("links", []):
        if not link["url"].startswith("http"):
            link["url"] = urljoin(base, link["url"])
    
    # Filter out anything that's still not a valid URL
    links["links"] = [l for l in links["links"] if l["url"].startswith("http")]
    
    print(f"Found {len(links['links'])} relevant links")
    return links

In [40]:
def fetch_page_and_all_relevant_links(url):
    content = fetch_website_contents(url)
    relevant_link = select_relevant_links(url)
    result = f"## Landing Page:\n\n{content}\n## Relevant Links:\n"
    for link in relevant_link['links']:
        result += f"\n\n### Link:{link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://coursera.org"))

Found 6 relevant links
## Landing Page:

Coursera | Courses, Professional Certificates, and Degrees Online

For
Individuals
For
Businesses
For
Universities
For
Governments
Explore
Degrees
​
Log In
Join for Free
Join for Free
Learn without limits
Meet new goals with midyear savings
Expand your skills with 10,000+ programs for ₹7,499/year—that's ₹625/month, billed annually.
Save now
Refresh skills. Recharge teams.
Save 40% and see how you can help your team develop skills without slowing them down.
Save 40% on Teams
Learn AI skills from leading names
Pracitical knowledge from OpenAI, Anthropic, Google, and more is here for your career path.
Join for Free
Previous
Next
Go to item 1
Go to item 2
Go to item 3
New and popular
Most popular
Hot new releases
Trending AI courses
Previous
Next
Get job-ready for an in-demand career
Get job-ready for an in-demand career
No prior experience needed to get started.
Explore programs
Data
Business
Sales & Marketing
IT
Software Engineering
Get midyear sa

In [41]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [42]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [48]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model="gemma3",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [49]:
create_brochure("HuggingFace", "https://huggingface.co")

Found 10 relevant links


## Hugging Face: The AI Community Building the Future

**A Platform for Collaboration & Innovation in Machine Learning**

Hugging Face is a vibrant community-driven platform empowering developers, researchers, and businesses to build innovative AI applications.  We provide the tools and resources needed to create, discover, and deploy cutting-edge models, datasets, and applications – all within a collaborative ecosystem. 

**What We Offer:**

* **A Massive Model Hub:** Explore over 2 million pre-trained machine learning models covering diverse tasks like natural language processing, computer vision, audio analysis, and more.  Trending models include:
    * `zai-org/GLM-5.2` (Large Language Model) – Over 67k+ downloads
    * `baidu/Unlimited-OCR` (Optical Character Recognition) – Over 70k+ downloads
    * Gemma family - A range of open-source, high-performing models

* **Datasets Galore:**  Access and contribute to a vast collection of over 500,000 datasets for training and experimenting with your own AI projects. 

* **Spaces:  Deploy & Share Your Apps:** Easily deploy and share your Machine Learning applications – including demos, interactive interfaces, and visualizations – directly through Hugging Face Spaces. Examples include:
   * Gemma 4 WebGPU Kernels - Chat with Gemma 4 E2B AI model in your browser
   * Pro Realism Edit Studio - Powerful image editing tool

* **HuggingChat & Collections:** Engage with the community and experiment with conversational AI models, organized through curated collections.



**Our Community: Driven by Collaboration**

Hugging Face’s core is its passionate and active community.  This community thrives on open collaboration, knowledge sharing, and constant innovation. Access to this community includes:

* **Discord & Forums:** Connect with peers, get support, and participate in discussions.
* **GitHub Repository:** Contribute to the development of our core technologies and tools. 
* **Daily Papers:** Stay at the forefront of research with access to recent publications and advancements.
* **Organizations:** Create a space within this platform for teams to deploy and share resources.

**For Customers & Investors:**

Hugging Face provides enterprise-grade solutions designed to accelerate your AI journey:

* **Enterprise Support:** Dedicated support options tailored to the needs of businesses.
* **Inference Providers & Endpoints:** Scale your ML applications with our robust infrastructure for hosting and serving models efficiently. 
* **Storage Buckets:** Securely store large datasets and model weights.



**Careers at Hugging Face: Join Our Mission!**

We’re looking for talented individuals to join our team. We offer exciting opportunities in areas such as:

* **Machine Learning Engineering:** Develop and deploy cutting-edge AI models and infrastructure.
* **Data Science:** Analyze data, create datasets, and build predictive models.
* **Software Development:** Contribute to the core platform and tools.
* **Community Management:** Engage and nurture our vibrant community.

**Learn More & Get Involved:** 

* **Website:** [https://huggingface.co/](https://huggingface.co/)
* **Hub:**  [https://huggingface.co/hub](https://huggingface.co/hub) 


Join the future of AI – join the Hugging Face community!

## Stream Animation(idk whats its called)
This basically gives a typewriter like animation
Just like how the data flows back to us when we use chatgpt.

In [50]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="gemma3",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [51]:
stream_brochure("Coursera", "https://coursera.org")

Found 7 relevant links


## Coursera: Unlock Your Potential – Globally

**A Brochure for Customers, Investors & Recruits**

**(Image: A diverse group of people collaborating on a digital screen, overlaid with the Coursera logo)**

**Coursera is revolutionizing education by providing access to world-class learning experiences for anyone, anywhere.** Founded in 2012 by Andrew Ng and Daphne Koller, we’re driven by a powerful vision – that learning should be a right, not a privilege.  We partner with over 375 leading universities and companies globally to deliver flexible, affordable, and job-relevant education through courses, Specializations, Professional Certificates, and degree programs.

**For Individuals: Invest in Yourself**

*   **Learn from the Best:** Access thousands of courses taught by renowned professors and industry experts from institutions like Google, IBM, Microsoft, Stanford University, and more.
*   **Expand Your Skills:** Explore offerings across a wide range of fields including Data Science, Artificial Intelligence, Business, Software Engineering, IT, Sales & Marketing, and Personal Development. 
*   **Career Momentum:**  Gain practical skills with our trending courses like Python or Data Analytics - designed to boost your career prospects.
*   **Affordable Learning:** Start with a free trial and save up to 40% on annual subscriptions – that’s just ₹625 per month! (Prices may vary by region)



**For Businesses: Empower Your Team**

*   **Upskill & Reskill:** Equip your workforce with the latest skills in high-demand areas like AI, Data Science and more.
*    **Save on Training Costs:**  Reduce training costs by up to 40% while simultaneously boosting team performance.
*   **Custom Solutions**: Coursera for Business provides access to a dedicated catalog of learning content tailored to your organization's needs and goals. Explore Google Career Collection programs for entry-level training

**For Universities & Governments: Transform Education**

*   **Expand Reach:**  Deliver your curriculum globally, reaching millions of learners through our robust online platform.
*   **Collaborative Innovation:** Partner with Coursera to leverage our technology and expertise in creating cutting-edge learning experiences.


**Company Culture - Our DNA:**

Coursera is built on a culture of innovation, collaboration, and impact. We're driven by the belief that education can change the world – we’ve fostered an environment where curiosity thrives and teams work together to tackle ambitious challenges. We value diversity, inclusion, and continuous learning! 



**Careers at Coursera: Join Our Mission**

We are seeking passionate individuals ready to shape the future of learning.  Coursera & Udemy have combined to create a massive powerhouse, focused on building skills for the AI era. 

*   **Opportunities:** Explore career paths in Product Development, Engineering, Marketing, Content Design and more.
*   **Impact:** Contribute to our mission of democratizing access to education and empowering millions worldwide. 


**Join the Coursera Community Today!**

[Link to Coursera Website: www.coursera.org]
[Social Media Links - LinkedIn, Twitter/X, Facebook] 

**(Small Logo of Coursera)** – *Learn Without Limits.*